# Model Swapping: da MobileNetV2 a VGG16

In questo notebook sostituiamo l'architettura MobileNetV2 con **VGG16** per la classificazione di immagini su ImageNet.
Testiamo il modello su 4 soggetti diversi — elefante africano, auto sportiva, pianoforte a coda e aquila calva —
visualizzando le **Top-5 predizioni** per ciascuno.

La scelta del modello non è neutra: ogni architettura ha il suo metodo `preprocess_input` specifico.
VGG16 **sottrae la media dei canali RGB** del dataset ImageNet, mentre MobileNetV2 scala i pixel in `[-1, 1]`.
Usare il preprocessing sbagliato produce predizioni insensate, indipendentemente dalla qualità dell'immagine.

---

## Struttura Notebook

1. **Setup e Configurazione** — import, costanti, impostazioni grafiche
2. **Caricamento del Modello** — VGG16 con pesi ImageNet, summary architettura
3. **Download e Visualizzazione delle Immagini** — recupero da URL, griglia di anteprima
4. **Classificazione** — preprocessing VGG16, Top-5 predizioni con grafico affiancato
5. **Analisi dei Risultati** — interpretazione per soggetto, nota critica sul preprocessing

---

## 1. Setup e Configurazione

In [ ]:
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib as mpl
from PIL import Image
from io import BytesIO

import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input, decode_predictions

# --- Costanti ---
IMG_SIZE = (224, 224)   # dimensione di input attesa da VGG16
TOP_K    = 5            # numero di predizioni da visualizzare

# Immagini di test: URL diretti da Wikimedia Commons
TEST_IMAGES = [
    {
        "url"  : "https://upload.wikimedia.org/wikipedia/commons/3/37/African_Bush_Elephant.jpg",
        "label": "Elefante africano"
    },
    {
        "url"  : "https://upload.wikimedia.org/wikipedia/commons/b/b8/Silver_Ferrari_Luxury_Sports_Car.jpg",
        "label": "Auto sportiva (Ferrari 488 GTB)"
    },
    {
        "url"  : "https://upload.wikimedia.org/wikipedia/commons/c/c8/Grand_piano_and_upright_piano.jpg",
        "label": "Pianoforte a coda"
    },
    {
        "url"  : "https://upload.wikimedia.org/wikipedia/commons/6/6d/Bald_eagle_FWS.jpg",
        "label": "Aquila calva"
    },
]

# --- Stile grafico ---
plt.style.use("seaborn-v0_8-whitegrid")
mpl.rcParams.update({
    "figure.dpi"        : 120,
    "figure.facecolor"  : "white",
    "font.family"       : "sans-serif",
    "font.size"         : 11,
    "axes.titlesize"    : 13,
    "axes.titleweight"  : "bold",
    "axes.labelsize"    : 11,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "legend.frameon"    : True,
    "legend.framealpha" : 0.9,
    "legend.fontsize"   : 10,
})

print("=" * 70)
print("SETUP COMPLETATO".center(70))
print("=" * 70)
print(f"  TensorFlow  : {tf.__version__}")
print(f"  NumPy       : {np.__version__}")
print(f"  GPU         : {tf.config.list_physical_devices('GPU')}")
print(f"  Soggetti    : {len(TEST_IMAGES)}")
print(f"  Top-K       : {TOP_K}")
print("=" * 70)

---

## 2. Caricamento del Modello

Carichiamo VGG16 con `weights='imagenet'` e `include_top=True`: manteniamo il classification head
originale (4096 → 4096 → 1000 neuroni) perché vogliamo classificare direttamente sulle 1000 classi
ImageNet, non estrarre feature.

In [ ]:
def load_model() -> tf.keras.Model:
    """Carica VGG16 pre-addestrato su ImageNet con classification head.

    Parametri
    ---------
    Nessuno.

    Ritorna
    -------
    model : tf.keras.Model
        Modello VGG16 con pesi ImageNet e top incluso.
    """
    print("Caricamento VGG16 (prima esecuzione: ~550 MB)...")
    model = VGG16(weights="imagenet", include_top=True)
    print(f"  Parametri totali      : {model.count_params():>12,}")
    print(f"  Parametri addestrabili: {sum(tf.size(w).numpy() for w in model.trainable_weights):>12,}")
    return model


print("=" * 70)
print("CARICAMENTO MODELLO".center(70))
print("=" * 70)
model = load_model()
print("=" * 70)
model.summary()

## 3. Download e Visualizzazione delle Immagini di Test

Scarichiamo le 4 immagini da Wikimedia Commons e le ridimensioniamo a 224×224.
Prima di classificare, le visualizziamo in griglia per verificare che il download sia andato
a buon fine e che il soggetto sia ben riconoscibile.

In [ ]:
def download_image(url: str) -> Image.Image:
    """Scarica un'immagine da URL e la restituisce come oggetto PIL ridimensionato.

    Parametri
    ---------
    url : str
        URL diretto all'immagine (jpg/png).

    Ritorna
    -------
    img : PIL.Image.Image
        Immagine RGB ridimensionata a IMG_SIZE.
    """
    # Header User-Agent necessario per evitare blocchi HTTP 403 da Wikimedia
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
        raw = response.read()
    return Image.open(BytesIO(raw)).convert("RGB").resize(IMG_SIZE)


def preprocess_image(img: Image.Image) -> np.ndarray:
    """Applica il preprocessing specifico di VGG16 a un'immagine PIL.

    Parametri
    ---------
    img : PIL.Image.Image
        Immagine RGB 224×224.

    Ritorna
    -------
    img_array : np.ndarray
        Array shape (1, 224, 224, 3) pronto per model.predict().
    """
    arr = np.expand_dims(np.array(img, dtype="float32"), axis=0)
    return preprocess_input(arr)   # sottrae la media RGB di ImageNet


# --- Download di tutte le immagini ---
print("=" * 70)
print("DOWNLOAD IMMAGINI DI TEST".center(70))
print("=" * 70)

for soggetto in TEST_IMAGES:
    print(f"  Scaricamento: {soggetto['label']}...", end=" ")
    soggetto["img"] = download_image(soggetto["url"])
    print("OK")

print("-" * 70)
print(f"  Download completato — {len(TEST_IMAGES)} immagini ({IMG_SIZE[0]}×{IMG_SIZE[1]} px ciascuna)")
print("=" * 70)

In [ ]:
def plot_image_grid(test_images: list[dict]) -> None:
    """Visualizza le immagini di test in una griglia 2×2.

    Parametri
    ---------
    test_images : list[dict]
        Lista di dizionari con chiavi 'label' e 'img' (PIL.Image).

    Ritorna
    -------
    None
    """
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    fig.suptitle(
        f"Immagini di test — {len(test_images)} soggetti ({IMG_SIZE[0]}×{IMG_SIZE[1]} px)",
        fontsize=14, fontweight="bold"
    )

    for ax, soggetto in zip(axes.flatten(), test_images):
        ax.imshow(soggetto["img"])
        ax.set_title(soggetto["label"], fontsize=12)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_image_grid(TEST_IMAGES)

### Osservazioni — Soggetti selezionati

Le 4 immagini coprono categorie semanticamente distinte e ben rappresentate in ImageNet:

| Soggetto | Classe ImageNet attesa | Caratteristica visiva distintiva |
|---|---|---|
| **Elefante africano** | `African_elephant`, `tusker` | Corpo massiccio, zanne, pelle grigia rugosa |
| **Ferrari 488 GTB** | `sports_car`, `racer` | Carrozzeria bassa, profilo aerodinamico |
| **Pianoforte a coda** | `grand_piano` | Cassa triangolare, tastiera bianco/nero |
| **Aquila calva** | `bald_eagle` | Testa bianca, corpo marrone, becco giallo |

Il caso più ambiguo è la Ferrari: ImageNet non ha una classe specifica per marca, la rete dovrà
ricondurla alla categoria generica `sports_car`.

---

## 4. Classificazione con VGG16

Per ogni immagine applichiamo il preprocessing specifico di VGG16, eseguiamo la predizione e
visualizziamo le Top-5 classi. Il layout affianca l'immagine originale al grafico a barre:
la barra rossa indica il Top-1, le blu le posizioni 2–5.

In [ ]:
def classify_image(model: tf.keras.Model, img: Image.Image) -> list[tuple]:
    """Preprocessa l'immagine ed esegue la predizione con VGG16.

    Parametri
    ---------
    model : tf.keras.Model
        Modello VGG16 caricato.
    img : PIL.Image.Image
        Immagine RGB 224×224 (non ancora preprocessata).

    Ritorna
    -------
    predictions : list[tuple]
        Lista di tuple (id_classe, nome_classe, probabilità) ordinate per score decrescente.
    """
    img_array = preprocess_image(img)
    preds     = model.predict(img_array, verbose=0)
    return decode_predictions(preds, top=TOP_K)[0]

In [ ]:
def plot_results(img: Image.Image, predictions: list[tuple], label: str) -> None:
    """Visualizza immagine originale e Top-K predizioni affiancate.

    Parametri
    ---------
    img : PIL.Image.Image
        Immagine ridimensionata a 224×224.
    predictions : list[tuple]
        Output di classify_image().
    label : str
        Etichetta descrittiva del soggetto (usata nel titolo).

    Ritorna
    -------
    None
    """
    classi      = [p[1].replace("_", " ") for p in predictions]
    probabilita = [float(p[2]) * 100       for p in predictions]
    colori      = ["#e05c5c" if i == 0 else "#5b8fd4" for i in range(len(classi))]

    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"VGG16 — {label}", fontsize=14, fontweight="bold")

    # Immagine originale
    ax_img.imshow(img)
    ax_img.set_title("Immagine originale (224×224)")
    ax_img.axis("off")

    # Barre orizzontali Top-K
    bars = ax_bar.barh(classi[::-1], probabilita[::-1], color=colori[::-1])
    ax_bar.set_xlabel("Probabilità (%)")
    ax_bar.set_title(f"Top-{TOP_K} predizioni")
    ax_bar.set_xlim(0, max(probabilita) * 1.28)

    for bar, prob in zip(bars, probabilita[::-1]):
        ax_bar.text(
            bar.get_width() + 0.4, bar.get_y() + bar.get_height() / 2,
            f"{prob:.1f}%", va="center", fontsize=9
        )

    legenda = [
        mpatches.Patch(color="#e05c5c", label="Top-1"),
        mpatches.Patch(color="#5b8fd4", label="Top 2–5"),
    ]
    ax_bar.legend(handles=legenda, loc="lower right")

    plt.tight_layout()
    plt.show()

In [ ]:
# --- Classificazione e visualizzazione per ogni soggetto ---
print("=" * 70)
print("CLASSIFICAZIONE — VGG16 Top-5".center(70))
print("=" * 70)

for soggetto in TEST_IMAGES:
    print(f"\n{'─' * 70}")
    print(f"  {soggetto['label']}")
    print(f"{'─' * 70}")

    predizioni = classify_image(model, soggetto["img"])

    print(f"  {'Posizione':<12} {'Classe':<28} {'Probabilità':>12}")
    print(f"{'─' * 70}")
    for i, (_, classe, prob) in enumerate(predizioni, start=1):
        marker = "  ← Top-1" if i == 1 else ""
        print(f"  Top-{i:<7} {classe.replace('_', ' '):<28} {prob * 100:>10.2f}%{marker}")

    plot_results(soggetto["img"], predizioni, soggetto["label"])

---

## 5. Analisi dei Risultati

### Elefante africano — Top-1: `African elephant` (87.30%)

La predizione è corretta con alta confidenza. La distribuzione delle Top-5 è coerente e interpretabile:
`tusker` (11.96%) è un termine alternativo per lo stesso animale, e `Indian elephant` (0.73%)
condivide la stessa silhouette. Le ultime due posizioni — `triceratops` (0.00%) e `water buffalo` (0.00%)
— suggeriscono che la rete ha attivato debolmente feature legate alla massa corporea e alle appendici
cefaliche, comuni a queste classi. La confidenza complessiva sull'elefante (Top-1 + Top-2) supera il 99%.

---

### Auto sportiva (Ferrari 488 GTB) — Top-1: `sports car` (67.91%)

La predizione è corretta, ma la distribuzione è la più frammentata del test.
`grille` (10.42%) è insolito: non è una classe di veicolo ma un componente — il frontale a griglia della
Ferrari ha evidentemente attivato feature specifiche per quell'elemento. `golfcart` (9.17%) è la
previsione più sorprendente: entrambi i veicoli condividono un profilo basso e una carrozzeria aperta,
il che suggerisce che la rete si sia parzialmente agganciata alla forma generale piuttosto che ai dettagli
semantici. `convertible` (4.87%) è plausibile, `minivan` (3.66%) meno.
Questo caso dimostra i limiti di ImageNet-1K di fronte a soggetti con caratteristiche visive polisemiche.

---

### Pianoforte a coda — Top-1: `grand piano` (99.18%)

La predizione più netta tra i soggetti non-animali: 99.18% con un gap abissale rispetto al secondo posto.
`upright` (0.82%) conferma che la rete ha rilevato la presenza del pianoforte verticale nell'immagine,
ma ha correttamente privilegiato il soggetto più prominente. Le posizioni 3–5 (`organ`, `printer`,
`desk`) raccolgono frazioni trascurabili e riflettono la forma rettangolare nera presente nella scena.

---

### Aquila calva — Top-1: `bald eagle` (99.91%)

Confidenza più alta del test: 99.91%, con solo 0.09% residuo distribuito su altri uccelli rapaci.
Le posizioni 2–5 (`kite`, `vulture`, `hornbill`, `albatross`) sono tutte uccelli di grandi dimensioni —
la rete ha identificato correttamente la categoria semantica in tutti e 5 gli slot. Il colore bianco
della testa e il contrasto con il corpo scuro rappresentano una firma visiva molto discriminante,
probabilmente ben rappresentata in ImageNet.

---

### Riepilogo

| Soggetto | Top-1 | Confidenza | Distribuzione |
|---|---|---|---|
| Elefante africano | `African elephant` | 87.30% | Concentrata su elefantidi |
| Auto sportiva | `sports car` | 67.91% | Frammentata, errori semantici parziali |
| Pianoforte a coda | `grand piano` | 99.18% | Quasi univoca |
| Aquila calva | `bald eagle` | 99.91% | Univoca, Top-5 tutti rapaci |

I soggetti con una firma visiva più univoca (aquila, pianoforte) ottengono confidenze vicine al 100%.
Il caso della Ferrari è il più istruttivo: una predizione tecnicamente corretta (`sports car`) ma con
una distribuzione che rivela incertezza strutturale, frutto della mancanza di una classe specifica
per auto da corsa ad alta prestazione in ImageNet-1K.

---

### Nota critica sul preprocessing

Tutti i risultati confermano che il `preprocess_input` importato da
`tensorflow.keras.applications.vgg16` è stato applicato correttamente.
Se si fosse usato quello di MobileNetV2 — che scala i pixel in `[-1, 1]` invece di sottrarre
la media RGB `[103.939, 116.779, 123.68]` — i valori di input sarebbero stati fuori dalla distribuzione
attesa dalla rete, producendo predizioni casuali o assurde indipendentemente dalla qualità dell'immagine.
Questo è uno degli errori più comuni nel model swapping.